In [15]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import time
from datetime import datetime
import undetected_chromedriver as uc
import random

# 업데이트 타임 추적 코드
def update_time():
    driver.refresh()
    try:
        mn = WebDriverWait(driver, 29).until(EC.element_to_be_clickable((By.CSS_SELECTOR, "section > div > span[data-mds='Typography']"))).text
    except:
        return 0
    mn = mn.replace("분 전", "")
    mn = int(mn)
    
    return mn

def scroll_down():
    for k in range(1):   #1 대신에 다른 숫자 넣으면 그만큼 스크롤 다운이 더 됩니다.
        body = driver.find_element("tag name", "body")
        body.send_keys(Keys.SPACE)

# url, 제품명, 브랜드 내용 수집하는 함수 | Length는 순위 입력
def prod_data(length):
    urls = []
    product = []
    brand = []
    
    while len(urls) < length:
    
        lst_url = driver.find_elements(By.CSS_SELECTOR, "div[data-viewport-type='window'] > div > div > div > div > div > div > a:nth-child(2)")
        lst_brand = driver.find_elements(By.CSS_SELECTOR, "div[data-viewport-type='window'] > div > div > div > div > div > div > a:nth-child(1) > p")
        lst_prod = driver.find_elements(By.CSS_SELECTOR, "div[data-viewport-type='window'] > div > div > div > div > div > div > a:nth-child(2) > p")
        
        for idx, i in enumerate(lst_url):
            try:
                tmp = i.get_attribute("href")
            except:
                continue
            tmp_brand = lst_brand[idx].text
            tmp_prod = lst_prod[idx].text
            if tmp not in urls:
                urls.append(tmp)
                brand.append(tmp_brand)
                product.append(tmp_prod)
        
        scroll_down()
        time.sleep(1.5)
        
    urls = urls[:length]
    product = product[:length]
    brand = brand[:length]
    
    return urls, product, brand

In [19]:
options = uc.ChromeOptions()

options.add_argument('--disable-gpu')
options.add_argument('--no-sandbox')

driver = uc.Chrome(options=options, version_main=144) 

driver.get("https://www.musinsa.com/main/musinsa/ranking")
time.sleep(random.uniform(2, 4))

WebDriverWait(driver, 20).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "button[data-extra-info='전체']"))
).click()

df_sw = 0

while 1:    
    if update_time() == 7:
        now = datetime.now()
        formated_time = now.strftime("%Y-%m-%d %H:%M:%S")
        
        urls, product, brand = prod_data(100)
        time.sleep(1)
        
        hearts = []
        for url in urls:
            driver.get(url)
            heart = WebDriverWait(driver, 20).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "div[class*='Like__Container']"))
            )
            time.sleep(2)
            heart = heart.text
            hearts.append(heart)
        time_lst = [formated_time for i in range(len(urls))]
        tmp_df = pd.DataFrame(zip(time_lst, brand, product, hearts, urls), columns=["TIME","BRAND","PRODUCT","LIKE","URL"])
        if df_sw == 0:
            raw_df = tmp_df
            df_sw = 1
        else:
            raw_df = pd.concat([raw_df, tmp_df])
            
        driver.get("https://www.musinsa.com/main/musinsa/ranking")
        time.sleep(random.uniform(2, 4))
        
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.CSS_SELECTOR, "button[data-extra-info='전체']"))
        ).click()
        
    else: 
        time.sleep(30)

KeyboardInterrupt: 

In [21]:
raw_df.to_excel("test.xlsx")